# Open AI Agents sdk
## sendgrid- to send sales cold email
## learn about functional_tool wrappers, tools vs handoffs

In [37]:
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from dotenv import load_dotenv
from typing import Dict
import sendgrid
from sendgrid import Content, Mail
import os
import asyncio

load_dotenv(override=True)

True

In [17]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

agent1 = Agent("Professional Sales Agent","", model="gpt-4o-mini")

agent2 =Agent("Engaging Sales Agent","", model="gpt-4o-mini")

agent3 =Agent("Busy Sales Agent","", model="gpt-4o-mini")

In [16]:

def send_email():
    sg = sendgrid.SendGridAPIClient(os.getenv("SEND_GRID_API_KEY"))
    from_email = "khera.aniruddh@gmail.com"
    to_email = "khera.aniruddh@gmail.com"
    content = Content("text/plain","This is a test email")
    mail =  Mail(from_email,to_email,"Test email", content).get()
    response = sg.send(mail)
    print(response.status_code)

send_email()


202


In [19]:
result =  Runner.run_streamed(agent1,instructions1)

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Simplify Your SOC 2 Compliance Process with ComplAI

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I represent ComplAI, a leading provider of AI-powered SaaS tools designed to streamline SOC 2 compliance and audit preparations for organizations like yours.

As you may know, achieving and maintaining SOC 2 compliance can be both challenging and resource-intensive. Our platform simplifies this process by automating documentation, monitoring compliance status in real-time, and providing actionable insights to ensure you're audit-ready at all times.

Key Benefits of ComplAI:

- **Automated Documentation**: Reduce time spent on manual processes with our intuitive interface.
- **Real-Time Monitoring**: Gain visibility into your compliance status, enabling proactive management.
- **Audit Preparedness**: Stay ready for audits with confidence through organized and accessible documentation.

I would love the opportunity to discuss how ComplAI 

In [32]:
async def parallel_cold_sales_email()->[]:
   with trace("Parallel sales cold emails"):
      results =await asyncio.gather(
      Runner.run(agent1,instructions1),
      Runner.run(agent2,instructions2),
      Runner.run(agent3,instructions3)
      )

   outputs= [result.final_output for result in results]
   return outputs

for output in outputs:
    print(output + "\n\n")



Certainly! Here’s a professional cold email template you can use to reach out to potential clients for ComplAI:

---

**Subject:** Streamline Your SOC 2 Compliance Process with ComplAI

Hi [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I’m reaching out from ComplAI, where we specialize in simplifying SOC 2 compliance through our advanced AI-powered SaaS tool.

In today’s regulatory environment, maintaining compliance can be a daunting task, often leading to confusion, delays, and unnecessary costs. Our platform is designed to not only streamline the compliance process but also ensure that your organization is well-prepared for audits, allowing you to focus on what truly matters—growing your business.

Some key features of ComplAI include:

- **Automated Compliance Checks:** Continuous monitoring for compliance gaps, allowing for real-time adjustments.
- **Document Management:** Effortlessly store and retrieve necessary documentation for audits.
- *

In [25]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

In [35]:
with trace("best sales cold email picker"):
    best = await Runner.run(sales_picker," ".join(await parallel_cold_sales_email()))

print(best.final_output)

Trace already exists. Creating a new trace, but this is probably a mistake.


Subject: Let’s Get Compliant – No Magic Wands Required! ✨

Hi [Recipient's Name],

I hope this email finds you riding the wave of productivity and not drowning in spreadsheets! 🏄‍♂️

I’m [Your Name] from ComplAI, where we believe that SOC2 compliance shouldn’t require a PhD in “Audit-ology.” Imagine a world where you could breeze through audits while sipping your favorite coffee, instead of sweating bullets at the mere thought of compliance! ☕💼

Our AI-powered tool is like having a compliance wizard on your team, minus the pointy hat (and the awkward wand twirling). With ComplAI, you can automate documentation, prepare for audits, and keep your team focused on what really matters – like solving the world’s problems (or perfecting that office coffee recipe).

Here’s the magic part: We help you turn chaos into compliance, so you can unlock the true potential of your business without sacrificing your sanity! 

Got 15 minutes this week? Let’s chat about how we can help you conquer complian

In [72]:
@function_tool
def send_email(msg: str):
    """Send out an email with the given body to all sales prospects."""
    sg = sendgrid.SendGridAPIClient(os.getenv("SEND_GRID_API_KEY"))
    from_email = "khera.aniruddh@gmail.com"
    to_email = "khera.aniruddh@gmail.com"
    content = Content("text/plain", msg)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    response = sg.send(mail)
    return {"status": response.status_code}

In [73]:
tool1 = agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool2 = agent2.as_tool(tool_name="sales_agent2", tool_description="Write a cold sales email")
tool3 = agent3.as_tool(tool_name="sales_agent3", tool_description="Write a cold sales email")

tools = [tool1, tool2, tool3, send_email]

tools


[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x125355b50>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._Failu

In [74]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""

sales_mail_selecter = Agent(name="sales mail selecter", instructions=instructions, tools=tools, model="gpt-4o-mini")


with trace("Sales manager"):
     result = await Runner.run(sales_mail_selecter, "Send a cold sales email addressed to 'Dear CEO'")


In [56]:
result.final_output

'The cold sales email has been successfully sent to the CEO. If you need any further assistance or adjustments, feel free to ask!'

## Handoffs

In [75]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."



subject_agent = Agent(name = "Email Subject writer", instructions= subject_instructions, model="gpt-4o-mini")
subject_tool = subject_agent.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_agent = Agent(name = "HTML email body converter",instructions= html_instructions, model="gpt-4o-mini")
html_tool = html_agent.as_tool(tool_name="html_email_converter" ,tool_description="HTML email body converter")

In [81]:
@function_tool
def send_html_email(msg: str, subject: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(os.getenv("SEND_GRID_API_KEY"))
    content = Content("text/html", msg)
    mail = Mail("khera.aniruddh@gmail.com","khera.aniruddh@gmail.com", subject=subject, html_content=content).get()
    sg.send(mail)
    return {"status": "OK"}


In [82]:

tools = [subject_tool, html_tool, send_html_email]

instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_email_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

email_agent= Agent(name="Email Manager",instructions= instructions, tools=tools, model="gpt-4o-mini", handoff_description="Convert an email to HTML and send it")



In [83]:
handoffs = [email_agent]
print(tools)
print(handoffs)

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x1252c0490>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False), FunctionTool(name='html_email_converter', description='HTML email body converter', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_

In [84]:

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = ("Send out a cold sales email addressed to Dear CEO from Alice")

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function c

# Guardrails

In [85]:
from pydantic import BaseModel

class NameChecker(BaseModel):
    is_name_included: bool
    name: str


In [ ]:
guard_rail_agent = Agent(name="Name Checker Guardrail Agent", 
    instructions="Check if the user is including someone's personal name in what they want you to do.", 
    output_type=NameChecker,
    model="gpt-4o-mini")
